# Automatic Mixer Calibration — QCM-RF / QRM-RF

Suppresses **LO leakage** ($\omega_{LO}$) and the **image sideband** ($\omega_{LO}-\omega_{NCO}$) using
the RF module's built-in calibration circuitry. No spectrum analyzer, no output cabling.

All the logic lives in [`calibrate_mixers.py`](calibrate_mixers.py) next to this notebook — this is
just the interactive front end. Everything here is equivalent to:

```
python scripts/calibrate_mixers.py <config_dir>
```

### Before you run

- **No `scqo` session / `HardwareAgent` may hold this cluster.** This connects directly through
  `qblox_instruments`. Restart this kernel (and any session kernel) when you are done.
- **Never mid-experiment.** The output switches go OFF during calibration and calibration
  interrupts *every* sequencer in the module. The script snapshots the running ones and restarts
  them, but treat this as an explicit calibration step.
- **Grounding matters for the result:** all modules screwed in top *and* bottom, every empty
  cluster slot filled with a screwed-in metal flow blocker.
- Expect **~35 dBc** suppression of both spurs at 30 % IF amplitude. That is the AMC design floor.
  If your spur budget needs more (image landing on a neighbour qubit or a readout resonator), fall
  back to manual calibration against an analyzer or a qubit-based probe.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "scripts" else Path.cwd() / "scripts"))
import calibrate_mixers as cm

CONFIG_DIR = Path(r"D:\qpu_data_dev\chipA\cd1\qblox\backend_config")
IF_AMP = cm.DEFAULT_IF_AMP  # per-path tone amplitude; the 35 dBc spec is quoted at 0.3

hw, dut = cm.load_configs(CONFIG_DIR)

## 1. Plan — what will be calibrated, from the config alone

`hw_config.json` gives slot &rarr; module type, the connectivity graph gives port &rarr; output, and
`modulation_frequencies` gives the LO. The NCO is `clock_freq - lo_freq` read off `dut_config.json`
(or the explicit `interm_freq` if the config has one).

Check the resulting **RF** column against the qubit / resonator you expect before going on.

In [ ]:
plan = cm.build_plan(hw, dut)
print(cm.describe_plan(plan))

Narrow it if you only want one module or one port-clock:

```python
plan = cm.build_plan(hw, dut, slots=[4])                          # QCM-RF drive only
plan = cm.build_plan(hw, dut, portclocks=["q1:res-q1.ro"])        # one readout line
plan = cm.build_plan(hw, dut, sequencer_map={"q1:res-q1.ro": 1})  # override the seq index
```

Sequencer indices mirror `qblox_scheduler`'s allocation (lowest free sequencer, port-clocks in
config order). With one port-clock per module that is sequencer 0 — exactly what the compiler uses.
On a multiplexed module, check it and override if needed: a sideband cal on the wrong sequencer is
silent.

Section 4 (`cm.diagnose`) works on **one** port-clock — the first in `plan.groups`. Narrow the plan
here first if you want to diagnose a different one.

## 2. Calibrate

Set `DUMMY = True` for a no-hardware rehearsal (the cal calls become no-ops but the whole control
flow, the sequencer restart and the cache all run).

The cache in `<config_dir>/mixer_cal.json` is validated **against the live hardware values**, not a
timestamp — a cluster reboot clears the corrections and shows up as a miss, so re-running this cell
is cheap and safe. Pass `force=True` to recalibrate regardless.

`CAL_ATT` is the output attenuation held **during** the calibration; whatever the module had is
restored afterwards. It defaults to 0 dB because the attenuator sits *after* the mixer, so an
operating value like 42 dB on a readout line can bury the image below the AMC detector. That is safe
because the RF output switch stays **open** the whole time — the detector is internal, Qblox opens
the switch during the cal anyway, and switch suppression is >60 dB, so nothing reaches the fridge.
Set `SWITCH_ON = True` only to watch the spurs on an analyzer (and raise `CAL_ATT` if you do).

In [ ]:
DUMMY = False
FORCE = False
CAL_ATT = 0        # dB held during the calibration, restored after; None = leave it alone
SWITCH_ON = False  # keep the RF output switch open while the tone plays
ATTEMPTS = cm._CAL_ATTEMPTS  # sideband_cal() only lands some of the time — this is the knob

cache_path = CONFIG_DIR / "mixer_cal.json"
cache = cm.load_cache(cache_path)

cluster = cm.open_cluster(plan, ip=None, dummy=DUMMY)
try:
    records = cm.calibrate_cluster(
        cluster, plan, amp=IF_AMP, cache=cache, force=FORCE,
        cal_att=CAL_ATT, switch_on=SWITCH_ON, attempts=ATTEMPTS,
    )
finally:
    cm.close_cluster(cluster)  # release the cluster for the next scqo session

records

## 3. Verify and log

Each record carries a `status`:

| status | meaning |
|---|---|
| `calibrated` | the values moved off the vendor defaults — cached, with the `attempts` it took |
| `cached` | the hardware still holds the calibration we recorded earlier |
| `no-op` | the cal ran, the firmware reported success, and **nothing changed** — not cached |

**`sideband_cal()` only lands some of the time.** On chipA it no-ops on roughly half to
three-quarters of calls, so `ATTEMPTS` retries until the values move; 2–4 attempts is normal
and a whole run exhausting 12 is not. Watch the `attempts` field — a rising trend across
cooldowns is the number to take to Qblox support.

A `no-op` that survives all attempts is a failure. Watch especially for the asymmetric case —
**LO fine, sideband `no-op`**: LO leakage is DC mixer feedthrough and calibrates happily with
no tone at all, so good LO offsets prove nothing. Section 4 A/Bs the conditions.

Everything is appended to `mixer_cal.json` under `history` for cooldown-to-cooldown drift
tracking. Reference values measured 2026-07-27: slot 4 (QCM-RF drive) ratio ≈ 1.0330, phase
≈ −6.99°; slot 8 (QRM-RF readout) ratio ≈ 1.0645, phase ≈ −23.3°.

In [ ]:
from collections import Counter
from datetime import datetime, timezone

for r in records:
    if r["status"] == "cached":
        print(f"{r['key']:<32} cached")
        continue
    flag = "MOVED" if r["status"] == "calibrated" else "NO-OP (still on the defaults)"
    print(f"{r['key']:<32} {r['before']} -> {r['after']}   {flag}")

counts = Counter(r["status"] for r in records)
print(f"\n{counts['calibrated']} calibrated, {counts['cached']} cached, {counts['no-op']} no-op")

cache["history"].append(
    {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "config_dir": str(CONFIG_DIR),
        "cluster": plan.cluster_name,
        "ip": None if DUMMY else plan.ip,
        "dummy": DUMMY,
        "amp": IF_AMP,
        "cal_att": CAL_ATT,
        "switch_on": SWITCH_ON,
        "records": records,
    }
)
cm.save_cache(cache_path, cache)
print(f"logged -> {cache_path}")

## 4. Diagnose a `no-op` (only if section 3 reported one)

Runs one port-clock through the suspects, one at a time, resetting `mixer_corr_*` to the defaults
before each trial so "did it move?" needs no interpretation. Trial **A** reproduces the original
failing behaviour; the first trial that moves names the cause. The sequencer state is printed for
every trial — if it never says `RUNNING`, the tone was never playing and attenuation is a red
herring.

Writes no cache, and restores attenuation, markers and the running sequencers.

In [ ]:
cluster = cm.open_cluster(plan, ip=None, dummy=DUMMY)
try:
    trials = cm.diagnose(cluster, plan, amp=IF_AMP)
finally:
    cm.close_cluster(cluster)

## When to re-run

| event | LO cal | sideband cal |
|---|---|---|
| LO retune (`modulation_frequencies[...].lo_freq`) | **yes** | **yes** (the IF moves with it) |
| IF retune only (`clock_freqs.f01` / `.readout`) | no | **yes** |
| drive/readout amplitude changed a lot | **yes** | **yes** (calibration is amplitude dependent) |
| cluster reboot / power cycle | **yes** | **yes** (the corrections are volatile) |
| new cooldown | **yes** | **yes** (and keep the log for drift) |

Re-running this notebook after any of those is enough — cached entries that still hold on the
hardware are skipped automatically.

### Does this survive a `scqo run`?

Yes, **as long as `hw_config.json` has no `hardware_options.mixer_corrections` block** (chipA does
not). The instrument coordinator only pushes `mixer_corr_gain_ratio` / `mixer_corr_phase_offset_degree`
and `out<k>_offset_path0/1` when a port-clock declares them, and it never resets the module — so with
no block, the AMC values stay.

If you *do* add a `mixer_corrections` entry, the next upload overwrites this calibration. In that case
hand the job to the scheduler instead of this notebook:

```json
"mixer_corrections": {
  "q1:mw-q1.01": {"auto_lo_cal": "on_lo_interm_freq_change",
                  "auto_sideband_cal": "on_interm_freq_change"}
}
```